<a href="https://colab.research.google.com/github/Faiq-danZ/worksafe-ai/blob/main/inference/predict_tabular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. MOUNT DRIVE

In [46]:
from google.colab import drive
drive.mount('/content/drive')

path_model     = '/content/drive/MyDrive/Data/Models/'
path_processed = '/content/drive/MyDrive/Data/Data/Processed/'
print("Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


## 2. IMPORT Library

In [47]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow import keras
print(f"TF: {tf.__version__}")

TF: 2.20.0


## 3. DEFINISI CUSTOM LOSS (WAJIB ADA SEBELUM LOAD MODEL)

In [48]:
class FocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, name='focal_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true    = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_true_oh = tf.cast(tf.one_hot(y_true, depth=3), tf.float32)
        y_pred    = tf.cast(y_pred, tf.float32)
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce           = -y_true_oh * tf.math.log(y_pred)
        p_t          = tf.reduce_sum(y_true_oh * y_pred, axis=-1, keepdims=True)
        focal_weight = tf.pow(1.0 - p_t, self.gamma)
        loss         = self.alpha * focal_weight * tf.reduce_sum(ce, axis=-1)
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha})
        return cfg

print("FocalLoss OK.")

FocalLoss OK.


## 4. LOAD MODEL & ARTIFACTS

In [49]:
model = keras.models.load_model(
    path_model + 'worksafe_model_v1.keras',
    custom_objects={'FocalLoss': FocalLoss}
)

with open(path_model + 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open(path_model + 'imputer.pkl', 'rb') as f:
    imputer = pickle.load(f)

with open(path_model + 'feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

LABEL_MAP = {0: 'Low Risk', 1: 'Medium Risk', 2: 'High Risk'}

# Simpan nilai median dari imputer untuk default input
median_values = {
    col: float(imputer.statistics_[i])
    for i, col in enumerate(feature_cols)
}

print("Model & artifacts loaded.")
print(f"Jumlah fitur: {len(feature_cols)}")

Model & artifacts loaded.
Jumlah fitur: 50


##  5. FUNGSI PREPROCESSING INPUT

In [50]:
def preprocess_input(input_dict):
    df = pd.DataFrame([input_dict])

    for col in feature_cols:
        if col not in df.columns:
            df[col] = np.nan

    df = df[feature_cols]

    df_imp = pd.DataFrame(
        imputer.transform(df),
        columns=feature_cols
    )

    df_scaled = pd.DataFrame(
        scaler.transform(df_imp),
        columns=feature_cols
    )

    return df_scaled.values.astype(np.float32)

print("preprocess_input OK.")

preprocess_input OK.


## 6. FUNGSI PREDIKSI

In [51]:
def predict_risk(input_dict):
    X     = preprocess_input(input_dict)
    proba = model.predict(X, verbose=0)[0]

    kelas = int(np.argmax(proba))
    label = LABEL_MAP[kelas]

    return {
        'risk_label'   : label,
        'risk_class'   : kelas,
        'confidence'   : round(float(proba[kelas]) * 100, 2),
        'probabilities': {
            'Low Risk'   : round(float(proba[0]) * 100, 2),
            'Medium Risk': round(float(proba[1]) * 100, 2),
            'High Risk'  : round(float(proba[2]) * 100, 2),
        }
    }

In [52]:
print(feature_cols)

['act_repairing_and_maintaining_mechanical_equipment', 'act_controlling_machines_and_processes', 'skl_operation_and_control', 'skl_equipment_maintenance', 'skl_troubleshooting', 'skl_repairing', 'skl_operations_monitoring', 'act_handling_and_moving_objects', 'act_inspecting_equipment,_structures,_or_materials', 'skl_equipment_selection', 'act_operating_vehicles,_mechanized_devices,_or_equipment', 'act_performing_general_physical_activities', 'act_repairing_and_maintaining_electronic_equipment', 'skl_quality_control_analysis', 'skl_speaking', 'skl_active_listening', 'skl_writing', 'skl_reading_comprehension', 'skl_social_perceptiveness', 'skl_active_learning', 'skl_persuasion', 'act_drafting,_laying_out,_and_specifying_technical_devices,_parts,_and_equipment', 'skl_service_orientation', 'skl_critical_thinking', 'skl_negotiation', 'act_establishing_and_maintaining_interpersonal_relationships', 'act_getting_information', 'skl_learning_strategies', 'skl_judgment_and_decision_making', 'act_

## 7. CONTOH INFERENSI

In [53]:
# Ganti key dan value sesuai nama fitur dari feature_cols
# Bisa cek: print(feature_cols) untuk lihat nama fitur

sample = {fc: np.random.uniform(1, 5) for fc in feature_cols}  # dummy random

result = predict_risk(sample)

print("=" * 40)
print("WorkSafe AI - Hasil Prediksi")
print("=" * 40)
print(f"Label      : {result['risk_label']}")
print(f"Kelas      : {result['risk_class']}")
print(f"Confidence : {result['confidence']}%")
print(f"\nProbabilitas:")
for lbl, prob in result['probabilities'].items():
    bar = '█' * int(prob / 5)
    print(f"  {lbl:<14}: {prob:>5}%  {bar}")

WorkSafe AI - Hasil Prediksi
Label      : High Risk
Kelas      : 2
Confidence : 97.54%

Probabilitas:
  Low Risk      :   0.0%  
  Medium Risk   :  2.45%  
  High Risk     : 97.54%  ███████████████████


In [54]:
print("=" * 50)
print("  WorkSafe AI - Cek Risiko Pekerjaan Kamu")
print("=" * 50)
print("Isi nilai untuk setiap skill (skala 1-5).")
print("1=Sangat Rendah, 3=Sedang, 5=Sangat Tinggi")
print("-" * 50)

def tanya(pertanyaan, nama_fitur):
    while True:
        try:
            val = float(input(f"{pertanyaan} (1-5): "))
            if 1.0 <= val <= 5.0:
                return nama_fitur, val
            print("  Angka harus antara 1 sampai 5.")
        except ValueError:
            print("  Masukkan angka saja.")

pertanyaan_list = [
    ("Kemampuan memperbaiki/merawat mesin",       "act_repairing_and_maintaining_mechanical_equipment"),
    ("Kemampuan mengoperasikan mesin/kendaraan",  "act_operating_vehicles,_mechanized_devices,_or_equipment"),
    ("Kemampuan kontrol peralatan",               "skl_operation_and_control"),
    ("Kemampuan perawatan peralatan",             "skl_equipment_maintenance"),
    ("Kemampuan troubleshooting",                 "skl_troubleshooting"),
    ("Kemampuan berpikir kritis",                 "skl_critical_thinking"),
    ("Kemampuan belajar mandiri",                 "skl_active_learning"),
    ("Kemampuan memecahkan masalah kompleks",     "skl_complex_problem_solving"),
    ("Kemampuan analisis sistem",                 "skl_systems_analysis"),
    ("Kemampuan komunikasi/berbicara",            "skl_speaking"),
    ("Kemampuan menulis",                         "skl_writing"),
    ("Kemampuan membaca & memahami teks",         "skl_reading_comprehension"),
    ("Kemampuan mengambil keputusan",             "skl_judgment_and_decision_making"),
    ("Kemampuan manajemen waktu",                 "skl_time_management"),
    ("Kemampuan bekerja dengan komputer",         "act_working_with_computers"),
    ("Kemampuan analisis data",                   "act_analyzing_data_or_information"),
    ("Kemampuan perencanaan & organisasi",        "act_organizing,_planning,_and_prioritizing_work"),
    ("Kemampuan negosiasi",                       "skl_negotiation"),
    ("Kemampuan koordinasi tim",                  "skl_coordination"),
    ("Kemampuan memantau proses/lingkungan",      "act_monitoring_processes,_materials,_or_surroundings"),
]

sample_input = {}
for pertanyaan, fitur in pertanyaan_list:
    fitur_key, nilai = tanya(pertanyaan, fitur)
    sample_input[fitur_key] = nilai

# Fitur yang tidak diisi pakai nilai median dari data training
for col in feature_cols:
    if col not in sample_input:
        sample_input[col] = median_values[col]

print("\nMemproses...")
result = predict_risk(sample_input)

print("\n" + "=" * 50)
print("  HASIL ANALISIS RISIKO PEKERJAAN KAMU")
print("=" * 50)
print(f"  Kategori Risiko : {result['risk_label']}")
print(f"  Confidence      : {result['confidence']}%")
print("-" * 50)
print("  Probabilitas tiap kategori:")
for lbl, prob in result['probabilities'].items():
    bar = '█' * int(prob / 5)
    print(f"  {lbl:<14}: {prob:>6}%  {bar}")
print("=" * 50)

if result['risk_class'] == 0:
    print("\n  Pekerjaan kamu tergolong AMAN dari risiko otomasi.")
    print("  Profil skill kamu didominasi kemampuan kognitif")
    print("  yang sulit digantikan mesin.")
elif result['risk_class'] == 1:
    print("\n  Pekerjaan kamu memiliki risiko SEDANG.")
    print("  Ada sebagian tugas yang berpotensi terotomasi,")
    print("  namun masih banyak aspek yang membutuhkan manusia.")
else:
    print("\n  Pekerjaan kamu memiliki risiko TINGGI terkena otomasi.")
    print("  Disarankan untuk mengembangkan skill kognitif")
    print("  dan digital agar tetap relevan di pasar kerja.")

  WorkSafe AI - Cek Risiko Pekerjaan Kamu
Isi nilai untuk setiap skill (skala 1-5).
1=Sangat Rendah, 3=Sedang, 5=Sangat Tinggi
--------------------------------------------------
Kemampuan memperbaiki/merawat mesin (1-5): 5
Kemampuan mengoperasikan mesin/kendaraan (1-5): 5
Kemampuan kontrol peralatan (1-5): 5
Kemampuan perawatan peralatan (1-5): 5
Kemampuan troubleshooting (1-5): 
  Masukkan angka saja.
Kemampuan troubleshooting (1-5): 
  Masukkan angka saja.
Kemampuan troubleshooting (1-5): 
  Masukkan angka saja.
Kemampuan troubleshooting (1-5): 
  Masukkan angka saja.
Kemampuan troubleshooting (1-5): 
  Masukkan angka saja.
Kemampuan troubleshooting (1-5): 
  Masukkan angka saja.
Kemampuan troubleshooting (1-5): 
  Masukkan angka saja.
Kemampuan troubleshooting (1-5): 
  Masukkan angka saja.
Kemampuan troubleshooting (1-5): 5
Kemampuan berpikir kritis (1-5): 5
Kemampuan belajar mandiri (1-5): 5
Kemampuan memecahkan masalah kompleks (1-5): 5
Kemampuan analisis sistem (1-5): 5
Kemampua

## 8. BATCH INFERENCE (OPSIONAL)

In [55]:
df_infer = pd.read_csv(path_processed + 'test_data.csv').head(10)

# Data test_data.csv sudah di-scale, langsung masuk model
# tidak perlu lewat preprocess_input
results = []
for _, row in df_infer.iterrows():
    X     = row[feature_cols].values.astype(np.float32).reshape(1, -1)
    proba = model.predict(X, verbose=0)[0]
    kelas = int(np.argmax(proba))
    results.append({
        'actual_label': LABEL_MAP[int(row['risk_label'])],
        'risk_label'  : LABEL_MAP[kelas],
        'confidence'  : round(float(proba[kelas]) * 100, 2),
    })

df_results = pd.DataFrame(results)
print("\nHasil batch inference (10 sampel pertama):")
display(df_results)


Hasil batch inference (10 sampel pertama):


,actual_label,risk_label,confidence
0,High Risk,High Risk,99.68
1,Low Risk,Low Risk,99.94
2,High Risk,High Risk,89.26
3,Medium Risk,Medium Risk,93.87
4,Medium Risk,Medium Risk,99.55
5,High Risk,High Risk,86.82
6,High Risk,High Risk,94.35
7,Low Risk,Low Risk,100.00
8,Low Risk,Low Risk,99.71
9,Low Risk,Low Risk,99.97
